## Example 1: Host an agent behind the AgentCore Runtime contract

In [ ]:
import importlib.metadata

from bedrock_agentcore import BedrockAgentCoreApp
from starlette.testclient import TestClient

# Example 1: @app.entrypoint becomes POST /invocations; GET /ping is the health check the runtime polls
assert importlib.metadata.version("bedrock-agentcore") == "1.25.4"
app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload):
    return {"answer": payload["prompt"].upper()}


with TestClient(app) as http:
    reply = http.post("/invocations", json={"prompt": "hello"})
    ping = http.get("/ping")
assert reply.status_code == 200 and reply.json() == {"answer": "HELLO"}, reply.text
assert ping.status_code == 200 and ping.json()["status"] == "Healthy", ping.json()
print("Example 1 OK:", reply.json(), ping.json()["status"])

## Example 2: Stream a response with an async generator

In [ ]:
from bedrock_agentcore import BedrockAgentCoreApp
from starlette.testclient import TestClient

# Example 2: an async-generator entrypoint is sent as server-sent events
app = BedrockAgentCoreApp()


@app.entrypoint
async def invoke(payload):
    for word in payload["prompt"].split():
        yield {"token": word}


with TestClient(app) as http:
    reply = http.post("/invocations", json={"prompt": "one two three"})
assert reply.headers["content-type"].startswith("text/event-stream"), reply.headers["content-type"]
assert [line for line in reply.text.splitlines() if line.startswith("data:")] and "three" in reply.text, reply.text
print("Example 2 OK:", reply.text.count("data:"), "events")

## Example 3: The request context carries the session id

In [ ]:
from bedrock_agentcore import BedrockAgentCoreApp, RequestContext
from starlette.testclient import TestClient

# Example 3: an entrypoint that takes a second argument receives the runtime's request context
app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload, context: RequestContext):
    return {"session": context.session_id}


with TestClient(app) as http:
    reply = http.post("/invocations", json={}, headers={"X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": "session-123"})
assert reply.json() == {"session": "session-123"}, reply.json()
print("Example 3 OK:", reply.json())

## Example 4: Background work makes the agent report busy

In [ ]:
from bedrock_agentcore import BedrockAgentCoreApp, PingStatus

# Example 4: while an async task is registered the health check says HealthyBusy, so the runtime keeps the session
app = BedrockAgentCoreApp()
assert app.get_current_ping_status() == PingStatus.HEALTHY
task_id = app.add_async_task("index-documents")
assert app.get_current_ping_status() == PingStatus.HEALTHY_BUSY
assert app.complete_async_task(task_id)
assert app.get_current_ping_status() == PingStatus.HEALTHY
app.force_ping_status(PingStatus.HEALTHY_BUSY)
assert app.get_current_ping_status() == PingStatus.HEALTHY_BUSY
app.clear_forced_ping_status()
print("Example 4 OK:", [s.value for s in PingStatus])

## Example 5: A custom ping handler

In [ ]:
from bedrock_agentcore import BedrockAgentCoreApp, PingStatus
from starlette.testclient import TestClient

# Example 5: @app.ping overrides the health decision
app = BedrockAgentCoreApp()
state = {"busy": True}
app.ping(lambda: PingStatus.HEALTHY_BUSY if state["busy"] else PingStatus.HEALTHY)
app.entrypoint(lambda payload: {"ok": True})
with TestClient(app) as http:
    first = http.get("/ping").json()["status"]
    state["busy"] = False
    second = http.get("/ping").json()["status"]
assert (first, second) == ("HealthyBusy", "Healthy"), (first, second)
print("Example 5 OK:", first, "->", second)